In [ ]:
def spin_model(dataset,
               mu_1, sigma_1, mu_tilt_1, sigma_tilt_1,
               mu_2, sigma_2, mu_3, sigma_3,
               eta_ref, m_cut,
               a_ref=0.3, cos_ref=1.0):

    a_1 = dataset["a_1"]
    a_2 = dataset["a_2"]
    cos_tilt_1 = dataset["cos_tilt_1"]
    cos_tilt_2 = dataset["cos_tilt_2"]
    m_1 = dataset["mass_1"]

    # Components as before
    comp1 = gwpop.utils.truncnorm(a_1, mu_1, sigma_1, 1, 0) * \
            gwpop.utils.truncnorm(a_2, mu_1, sigma_1, 1, 0) * \
            gwpop.utils.truncnorm(cos_tilt_1, mu_tilt_1, sigma_tilt_1, 1, -1) * \
            gwpop.utils.truncnorm(cos_tilt_2, mu_tilt_1, sigma_tilt_1, 1, -1)

    comp2 = gwpop.utils.truncnorm(a_1, mu_2, sigma_2, 1, 0) * \
            gwpop.utils.truncnorm(a_2, mu_2, sigma_2, 1, 0) * \
            0.5 * 0.5

    comp3 = gwpop.utils.truncnorm(a_1, mu_3, sigma_3, 1, 0) * \
            gwpop.utils.truncnorm(a_2, mu_3, sigma_3, 1, 0) * \
            0.5 * 0.5

    # --- Anchor densities for comp1 and comp2 ---
    c1_ref = gwpop.utils.truncnorm(a_ref, mu_1, sigma_1, 1, 0) * \
             gwpop.utils.truncnorm(a_ref, mu_1, sigma_1, 1, 0) * \
             gwpop.utils.truncnorm(cos_ref, mu_tilt_1, sigma_tilt_1, 1, -1) * \
             gwpop.utils.truncnorm(cos_ref, mu_tilt_1, sigma_tilt_1, 1, -1)

    c2_ref = gwpop.utils.truncnorm(a_ref, mu_2, sigma_2, 1, 0) * \
             gwpop.utils.truncnorm(a_ref, mu_2, sigma_2, 1, 0) * \
             0.5 * 0.5

    f_ref = sigmoid(eta_ref)                 # unconstrained -> (0,1)
    weight_a = anchored_weight(f_ref, c1_ref, c2_ref)

    # Mass transition (your original; equivalent to sigmoid(m_1 - m_cut))
    zeta = 1 / (1 + xp.exp(-(m_1 - m_cut)))

    return (1 - zeta) * (weight_a * comp1 + (1 - weight_a) * comp2) + zeta * comp3